## setup
first install required Python packages (run in your environment):

```bash
pip install 'elasticsearch>=9.5,<10' requests pandas numpy sentence-transformers plotly networkx
```


**Version note.** The Python client version is not the Elasticsearch server version. The server reports its bundled Lucene version via `es.info()`.

In [25]:
!pip install 'nbformat>4.2.0' 'elasticsearch>=9.5,<10' requests pandas numpy sentence-transformers plotly networkx


[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [1]:
from elasticsearch import Elasticsearch
import requests
import time
import pandas as pd
import numpy as np

ES_HOST = 'http://localhost:9200'  # local Elasticsearch
es = Elasticsearch(ES_HOST)

# Record the server and bundled Lucene versions (distinct from the Python client).


def check_es():
    try:
        print('Elasticsearch cluster info:')
        print(es.info())
    except Exception as e:
        print('Error connecting to Elasticsearch:', e)


check_es()

Elasticsearch cluster info:
{'name': 'es01', 'cluster_name': 'es-docker-cluster', 'cluster_uuid': '0ifEmjNAQ_mZyAJ3AYcsHA', 'version': {'number': '9.5.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '8d4246a64bc255212407b1b313fe402391299c88', 'build_date': '2026-07-29T23:38:48.847847767Z', 'build_snapshot': False, 'lucene_version': '10.5.0', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


## Create an index


In [2]:
index_name = 'papers'

# define index mapping
mapping = {
    'mappings': {
        'properties': {
            'title': {'type': 'text'},
            'abstract': {'type': 'text',
                         'analyzer': 'standard',
                         'search_analyzer': 'english'},
            'year': {'type': 'integer'},
            'category': {'type': 'keyword'}
        }
    }
}

# create index (delete only this teaching index so the notebook is repeatable)
if es.indices.exists(index=index_name):
    es.indices.delete(index=index_name)
es.indices.create(index=index_name, body=mapping)
print('Created index', index_name)

Created index papers


# Fetch papers from arxiv

In [3]:
from xml.etree import ElementTree as ET

ARXIV_API = 'http://export.arxiv.org/api/query'

def fetch_arxiv_category(query, max_records=10000, batch=100):
    records = []
    start = 0
    while len(records) < max_records:
        to_fetch = min(batch, max_records - len(records))
        params = {
            'search_query': query,
            'start': start,
            'max_results': to_fetch,
        }
        r = requests.get(ARXIV_API, params=params, timeout=30)
        if r.status_code != 200:
            print('API error', r.status_code, ' - sleeping and retrying')
            time.sleep(5)
            continue
        root = ET.fromstring(r.text)
        ns = {'atom': 'http://www.w3.org/2005/Atom'}
        entries = root.findall('atom:entry', ns)
        if not entries:
            break
        for e in entries:
            aid = e.find('atom:id', ns).text.strip()
            title = e.find('atom:title', ns).text.strip().replace('\n', ' ')
            abstract = e.find('atom:summary', ns).text.strip().replace('\n', ' ')
            updated = e.find('atom:updated', ns).text
            # categories
            cats = [c.attrib.get('term') for c in e.findall('atom:category', ns)]
            records.append({'id': aid.split('/')[-1], 'title': title, 'abstract': abstract, 'updated': updated,
                            'categories': ','.join(cats)})
        start += to_fetch
        time.sleep(0.5) # polite delay
    return records


query = 'cat:cs.CL'
print('Starting fetch (this may take several minutes)...')

records = fetch_arxiv_category(query, max_records=1000, batch=300)
print('Fetched', len(records), 'records')

Starting fetch (this may take several minutes)...
Fetched 1000 records


In [4]:
def preprocess_records(records):
    df = pd.DataFrame(records)
    # extract year from updated
    df['year'] = pd.to_datetime(df['updated']).dt.year
    df['category'] = df['categories'].str.split(',').str[0]
    df = df[['id', 'title', 'abstract', 'year', 'category']]
    df = df.dropna(subset=['abstract'])
    df = df.drop_duplicates(subset=['title', 'abstract'])
    return df


df = preprocess_records(records)
print(df.head())

             id                                              title  \
0  2012.10055v2  End-to-End Speaker Diarization as Post-Processing   
1  2012.10187v1  Regularized Attentive Capsule Network for Over...   
2  2012.10226v1  Should I visit this place? Inclusion and Exclu...   
3  2012.13004v1  Speech Synthesis as Augmentation for Low-Resou...   
4  2012.13190v2  QUACKIE: A NLP Classification Task With Ground...   

                                            abstract  year category  
0  This paper investigates the utilization of an ...  2020  eess.AS  
1  Distantly supervised relation extraction has b...  2020    cs.CL  
2  Although several automatic itinerary generatio...  2020    cs.IR  
3  Speech synthesis might hold the key to low-res...  2020    cs.CL  
4  NLP Interpretability aims to increase trust in...  2020    cs.CL  


# helper function to index from a pandas dataframe

In [5]:
from elasticsearch import helpers

def bulk_index_dataframe(df, index_name, batch_size=500):
    actions = []
    for _, row in df.iterrows():
        doc = {'title': row['title'], 'abstract': row['abstract'], 'year': int(row['year']),
               'category': row['category']}
        actions.append({'_index': index_name, '_id': row['id'], '_source': doc})
        if len(actions) >= batch_size:
            helpers.bulk(es, actions)
            actions = []
    if actions:
        helpers.bulk(es, actions)
    es.indices.refresh(index=index_name)
    print('Indexed', len(df), 'docs into', index_name)


## Index sample documents


In [6]:
bulk_index_dataframe(df, index_name)

es.indices.refresh(index=index_name)
print('Indexed and refreshed')

Indexed 1000 docs into papers
Indexed and refreshed


# sample queries

In [7]:
def search(query):
    res = es.search(index=index_name, body=query)
    # Print results
    print('Query results:')
    for i, hit in enumerate(res["hits"]["hits"]):
        res_doc = hit["_source"]
        print(f'{i}:{res_doc["title"]} ({hit["_score"]})')

In [8]:
search({"query": {"match": {"abstract": "machine learning"}}})

Query results:
0:Toward a Thermodynamics of Meaning (3.9574788)
1:Question Generation from SQL Queries Improves Neural Semantic Parsing (3.5286381)
2:Probing Neural Dialog Models for Conversational Understanding (3.3644428)
3:Adversarial Training for Satire Detection: Controlling for Confounding Variables (3.214849)
4:Interpretable Convolutional Filters with SincNet (3.0349257)
5:Knowledge-Enhanced Personalized Review Generation with Capsule Graph Neural Network (2.9523108)
6:A Variational Approach to Unsupervised Sentiment Analysis (2.9523108)
7:Discovering User Groups for Natural Language Generation (2.900113)
8:R-VQA: Learning Visual Relation Facts with Semantic Attention for Visual Question Answering (2.8740742)
9:Partially Shuffling the Training Data to Improve Language Models (2.86185)


In [9]:
query = {
    "query": {
        "multi_match": {
            "query": "deep learning",
            "fields": ["title", "abstract"]
        }
    }
}
search(query)

Query results:
0:Deep Learning Models for Automatic Summarization (6.2876353)
1:Deep Reinforcement Learning for Chinese Zero pronoun Resolution (5.72119)
2:Deep Representation Learning for Clustering of Health Tweets (5.72119)
3:Cross-lingual Short-text Matching with Deep Learning (5.72119)
4:Deep Reinforcement Learning for On-line Dialogue State Tracking (5.4745903)
5:Multimodal deep learning for short-term stock volatility prediction (5.4745903)
6:Improving Automatic Source Code Summarization via Deep Reinforcement Learning (5.4745903)
7:Named-Entity Linking Using Deep Learning For Legal Documents: A Transfer Learning Approach (5.4438605)
8:Morphological Skip-Gram: Using morphological knowledge to improve word representation (5.0440407)
9:DeL-haTE: A Deep Learning Tunable Ensemble for Hate Speech Detection (5.040105)


In [10]:
# aggregations
agg_query = {"aggs": {
    "publication_years": {
        "histogram": {
            "field": "year",
            "interval": 5
        }
    }
}}
res = es.search(index=index_name, body=agg_query)
for bucket in res["aggregations"]["publication_years"]["buckets"]:
    print(bucket["key"], bucket["doc_count"])

2015.0 537
2020.0 463


# update mappings for embeddings

In [11]:
mapping_update = {
    "properties": {
        "embedding": {
            "type": "dense_vector",
            "dims": 384,
            "index": True,
            "similarity": "dot_product",
            "index_options": {
                "type": "hnsw"
            }
        }
    }
}

es.indices.put_mapping(index=index_name, body=mapping_update)

ObjectApiResponse({'acknowledged': True})

### generate embeddings locally with SentenceTransformers

you can generate embeddings locally and index them into Elasticsearch.
This example uses `sentence-transformers/all-MiniLM-L6-v2` (384 dims).

In [12]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def local_embed(texts):
    if isinstance(texts, str):
        texts = [texts]
    arr = model.encode(texts, show_progress_bar=False)
    return [x.tolist() for x in arr]

print('Embedding a sample sentence (len =', len(local_embed('hello')), ')')

/Users/tteofili/dev/elasticsearch-benchmarks/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1106 08:22:36.555000 32135 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8670.09it/s]


Embedding a sample sentence (len = 1 )


# add embeddings

In [13]:
def bulk_index_dataframe_embed(df, index_name, batch_size=500):
    actions = []
    for _, row in df.iterrows():
        doc = {'title': row['title'], 'abstract': row['abstract'], 'year': int(row['year']),
               'category': row['category'], 'embedding': local_embed(row['abstract'])[0]}
        actions.append({'_index': index_name, '_id': row['id'], '_source': doc})
        if len(actions) >= batch_size:
            helpers.bulk(es, actions)
            actions = []
    if actions:
        helpers.bulk(es, actions)
    es.indices.refresh(index=index_name)
    print('Indexed', len(df), 'docs into', index_name)

In [14]:
bulk_index_dataframe_embed(df, index_name)

es.indices.refresh(index=index_name)
print('Indexed and refreshed')

Indexed 1000 docs into papers
Indexed and refreshed


## Vector search (knn)

**Implementation note:** this notebook exercises Elasticsearch's KNN API. The Lucene 10.6.0 implementation details (vector fields, HNSW readers/writers, collectors and quantized vector formats) are covered in the accompanying Java/API diagrams. The server's bundled Lucene version must be read from `es.info()` rather than inferred from the Python client dependency.


In [15]:
query_text = 'dense retrieval'
qvec = local_embed(query_text)[0]

knn_query = {
    'knn': {
        'field': 'embedding',
        'query_vector': qvec,
        'k': 5,
        'num_candidates': 10
    }
}

search(knn_query)

Query results:
0:Is Retriever Merely an Approximator of Reader? (0.7753174)
1:Open-Retrieval Conversational Question Answering (0.7423381)
2:Distilling Dense Representations for Ranking using Tightly-Coupled Teachers (0.7422905)
3:Fine-Grained Relevance Annotations for Multi-Task Document Ranking and Question Answering (0.7326656)
4:Neural document expansion for ad-hoc information retrieval (0.7139143)


## Hybrid search (BM25 + knn)

In [16]:
hybrid_query = {
    "query": {
        "bool": {
            "must": [{"match": {"title": query_text}}],
            "should": [
                {
                    "knn": {
                        "field": "embedding",
                        "query_vector": qvec,
                        "k": 5,
                        "boost": 2.0
                    }
                }
            ]
        }
    }
}

search(query=hybrid_query)

Query results:
0:Distilling Dense Representations for Ranking using Tightly-Coupled Teachers (7.8003025)
1:Open-Retrieval Conversational Question Answering (7.489955)
2:Dense Information Flow for Neural Machine Translation (6.9115353)
3:Neural document expansion for ad-hoc information retrieval (6.6215954)
4:Cross-lingual Retrieval for Iterative Self-Supervised Training (5.1937666)
5:COCO-CN for Cross-Lingual Image Tagging, Captioning and Retrieval (4.7645354)
6:FAQ Retrieval using Query-Question Similarity and BERT-Based Query-Answer Relevance (4.4008355)
7:COVID-19 Knowledge Graph: Accelerating Information Retrieval and Discovery for Scientific Literature (4.4008355)
8:Sanity Check: A Strong Alignment and Information Retrieval Baseline for Question Answering (4.4008355)
9:Improving Commonsense Question Answering by Graph-based Iterative Retrieval over Multiple Knowledge Sources (4.239042)


# Implementation visuals

These visualizations use the **same arXiv paper DataFrame** (`df`) indexed above.


In [17]:
from collections import Counter, defaultdict
import math
import plotly.express as px
import plotly.graph_objects as go

# Canonical small, deterministic subset for fast classroom visualizations.
papers_viz = df.dropna(subset=['title', 'abstract']).drop_duplicates('id').head(40).copy()
papers_viz['doc_id'] = range(len(papers_viz))
print(f'Visualization corpus: {len(papers_viz)} papers')


Visualization corpus: 40 papers


## 1. Inverted-index explorer

A simplified term → postings view. Terms are derived from titles and abstracts; positions, offsets, payloads and segment-local doc IDs are intentionally omitted. The goal is to connect the logical postings structure to Lucene's field/term APIs.


In [18]:
import re
def tokenize_demo(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())

postings = defaultdict(list)
for row in papers_viz.itertuples():
    terms = set(tokenize_demo(row.title + ' ' + row.abstract))
    for term in terms:
        postings[term].append(int(row.doc_id))

term_options = sorted(postings, key=lambda t: (-len(postings[t]), t))[:60]
term_df = pd.DataFrame({'term': term_options,
                        'document_frequency': [len(postings[t]) for t in term_options],
                        'postings': [', '.join(map(str, postings[t])) for t in term_options]})
display(term_df.head(20))
fig = px.bar(term_df.head(25), x='term', y='document_frequency',
             hover_data=['postings'], title='Most frequent terms and their document frequency')
fig.update_layout(xaxis_tickangle=-45)
fig.show()


,term,document_frequency,postings
0,and,40,"0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, ..."
1,of,40,"0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, ..."
2,in,39,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14,..."
3,the,39,"0, 1, 2, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14,..."
4,to,38,"0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, ..."
5,we,38,"0, 1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14,..."
6,a,37,"0, 1, 2, 3, 4, 5, 6, 7, 8, 11, 12, 13, 14, 15,..."
7,for,36,"0, 1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 13, 14, 15,..."
8,this,34,"0, 1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14,..."
9,is,31,"0, 2, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, ..."


## 2. Segment lifecycle simulator

This is a deliberately simplified lifecycle model: buffered documents become segments on flush; a merge combines selected segments into a new segment.


In [20]:
segment_states = [
    {'stage':'After flush 1', 'segments':[('S0', 0, 9)]},
    {'stage':'After flush 2', 'segments':[('S0', 0, 9), ('S1', 10, 19)]},
    {'stage':'After flush 3', 'segments':[('S0', 0, 9), ('S1', 10, 19), ('S2', 20, 24)]},
    {'stage':'After merge', 'segments':[('S3', 0, 24)]},
]
def plot_segment_state(state):
    fig = go.Figure()
    for y, (name, lo, hi) in enumerate(state['segments']):
        fig.add_trace(go.Bar(x=[hi-lo+1], y=[name], orientation='h', base=[lo],
                             name=name, hovertemplate=f'{name}: doc IDs {lo}–{hi}<extra></extra>'))
    fig.update_layout(title=state['stage'], xaxis_title='Document-ID range',
                      showlegend=False, barmode='overlay', height=260)
    fig.show()
for state in segment_states:
    plot_segment_state(state)


## 3. Query execution path: Lucene class interaction map

A conceptual class/role map for the search path.

In [21]:
nodes = [
    ('Query', 0, 2), ('IndexSearcher', 1, 2), ('Weight', 2, 2),
    ('Scorer / BulkScorer', 3, 2), ('Collector', 4, 2), ('TopDocs', 5, 2),
    ('IndexReader', 1, 0), ('LeafReaderContext', 2, 0), ('Similarity', 3, 0)
]
edges = [('Query','IndexSearcher'), ('IndexSearcher','Weight'), ('Weight','Scorer / BulkScorer'),
         ('Scorer / BulkScorer','Collector'), ('Collector','TopDocs'),
         ('IndexReader','IndexSearcher'), ('IndexReader','LeafReaderContext'),
         ('Similarity','Weight')]
pos = {n:(x,y) for n,x,y in nodes}
fig = go.Figure()
for a,b in edges:
    x0,y0 = pos[a]; x1,y1 = pos[b]
    fig.add_trace(go.Scatter(x=[x0,x1], y=[y0,y1], mode='lines',
                             line=dict(width=1.5), hoverinfo='skip', showlegend=False))
for name,x,y in nodes:
    fig.add_trace(go.Scatter(x=[x], y=[y], mode='markers+text', text=[name],
                             textposition='bottom center', marker=dict(size=28),
                             showlegend=False, hovertemplate=f'{name}<extra></extra>'))
fig.update_layout(title='Lucene search abstractions (conceptual)', xaxis_visible=False,
                  yaxis_visible=False, height=420)
fig.show()


## 4. BM25 score inspection (teaching implementation)

The following computes a simplified BM25 score over the visualization corpus. It is useful for exposing TF, document frequency, document length and average length.


In [23]:
k1, b = 1.2, 0.75
token_docs = [tokenize_demo(r.title + ' ' + r.abstract) for r in papers_viz.itertuples()]
N = len(token_docs)
avgdl = sum(map(len, token_docs)) / max(N, 1)
dfreq = Counter(t for doc in token_docs for t in set(doc))
def bm25_demo(query, doc_tokens):
    score = 0.0; dl = len(doc_tokens); tfc = Counter(doc_tokens)
    for term in tokenize_demo(query):
        n_q = dfreq.get(term, 0)
        if not n_q: continue
        idf = math.log(1 + (N - n_q + 0.5) / (n_q + 0.5))
        tf = tfc.get(term, 0)
        score += idf * (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * dl / avgdl)) if tf else 0
    return score
demo_query = 'information retrieval'
scores = [bm25_demo(demo_query, toks) for toks in token_docs]
ranking = papers_viz[['doc_id','title']].copy(); ranking['bm25_demo'] = scores
ranking = ranking.sort_values('bm25_demo', ascending=False).head(12)
display(ranking)
#px.bar(ranking, x='bm25_demo', y='title', orientation='h',
#      title=f'Simplified BM25 ranking for: {demo_query}').show()


,doc_id,title,bm25_demo
8,8,Neural document expansion for ad-hoc informati...,6.303370
13,13,FAQ Retrieval using Query-Question Similarity ...,5.947922
32,32,Learning Attention-based Embeddings for Relati...,1.857287
36,36,NNE: A Dataset for Nested Named Entity Recogni...,1.603204
26,26,Semantic Matching of Documents from Heterogene...,1.571302
10,10,Query Expansion for Cross-Language Question Re...,1.467043
17,17,Who wrote this book? A challenge for e-commerce,1.310522
22,22,"Short Text Topic Modeling Techniques, Applicat...",1.204830
2,2,Should I visit this place? Inclusion and Exclu...,1.204830
29,29,A Review of Automated Speech and Language Feat...,1.164227


## 5. HNSW graph intuition

A small synthetic proximity graph to illustrate graph traversal. It is not built from the paper embeddings and is not Lucene's actual graph; use it to explain entry points, neighbor expansion and why `num_candidates` affects the recall/latency trade-off.


In [24]:
import networkx as nx
G = nx.random_geometric_graph(28, radius=0.34, seed=7)
xy = nx.get_node_attributes(G, 'pos')
edge_x, edge_y = [], []
for u,v in G.edges():
    edge_x += [xy[u][0], xy[v][0], None]; edge_y += [xy[u][1], xy[v][1], None]
fig = go.Figure()
fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines', hoverinfo='skip', showlegend=False))
fig.add_trace(go.Scatter(x=[xy[n][0] for n in G.nodes()], y=[xy[n][1] for n in G.nodes()],
                         mode='markers+text', text=[str(n) for n in G.nodes()],
                         textposition='top center', marker=dict(size=11), showlegend=False))
fig.update_layout(title='Synthetic proximity graph (HNSW intuition)', xaxis_visible=False,
                  yaxis_visible=False, height=480)
fig.show()
